# 12 — Conversational RAG with Persistent Memory

This notebook combines **RAG retrieval** with **persistent agent memory** for a conversational assistant that:

1. **Remembers** facts from previous conversations (extraction → consolidation)
2. **Retrieves** from both knowledge base AND memory during conversations
3. **Learns** new facts from each conversation turn
4. **Deduplicates** overlapping assertions via consolidation

| Component | SochDB Feature |
|---|---|
| Knowledge Base | Vector collection with hybrid search |
| Memory Store | ExtractionPipeline + Consolidator |
| Semantic Cache | cache_put / cache_get for repeated queries |
| Conversation History | KV store with transaction safety |

In [13]:
import os, json, time, shutil, hashlib
from openai import OpenAI
from sochdb import Database
from sochdb.namespace import CollectionConfig
from sochdb.memory.extraction import (
    ExtractionPipeline, ExtractionResult, Entity, Relation, Assertion,
)
from sochdb.memory.consolidation import (
    Consolidator, ConsolidationConfig, RawAssertion, CanonicalFact,
)
from sochdb.memory.retrieval import (
    HybridRetriever, RetrievalConfig, AllowedSet,
)

client = OpenAI(
    api_key=os.environ.get("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

EMBED_MODEL = "gemini-embedding-001"
CHAT_MODEL  = "gemini-2.5-flash-lite"
DIM = 3072

def embed(text: str) -> list[float]:
    resp = client.embeddings.create(model=EMBED_MODEL, input=text)
    return resp.data[0].embedding

def embed_batch(texts: list[str]) -> list[list[float]]:
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [d.embedding for d in resp.data]

print("Setup complete")

Setup complete


## 1. Initialize Database, Collections, and Memory Pipeline

In [14]:
DB_PATH = "./conv_rag_memory_db"
if os.path.exists(DB_PATH):
    shutil.rmtree(DB_PATH)

db = Database.open(DB_PATH)
ns = db.create_namespace("conv_rag")

# --- Knowledge base collection ---
kb_config = CollectionConfig(name="knowledge", dimension=DIM)
kb_collection = ns.create_collection(kb_config)

# --- Memory collection (for storing learned assertions) ---
mem_config = CollectionConfig(name="memory", dimension=DIM)
mem_collection = ns.create_collection(mem_config)

# --- Extraction pipeline (for extracting facts from conversations) ---
extraction_pipeline = ExtractionPipeline.from_database(
    db, namespace="conv_rag", collection="memory",
    embed_fn=embed,
)

# --- Consolidator (for merging/deduplicating facts) ---
consolidator = Consolidator.from_database(
    db, namespace="conv_rag",
    config=ConsolidationConfig(
        similarity_threshold=0.85,
        min_confidence=0.5,
        embedding_dim=DIM,
    ),
)

# --- HybridRetriever for knowledge base ---
kb_retriever = HybridRetriever.from_database(
    db, namespace="conv_rag", collection="knowledge",
    config=RetrievalConfig(k=3, alpha=0.7),
)

# --- HybridRetriever for memory ---
mem_retriever = HybridRetriever.from_database(
    db, namespace="conv_rag", collection="memory",
    config=RetrievalConfig(k=3, alpha=0.8),
)

print("Initialized: KB collection, Memory collection, Extraction pipeline, Consolidator, 2x Retrievers")

Initialized: KB collection, Memory collection, Extraction pipeline, Consolidator, 2x Retrievers


## 2. Seed Knowledge Base

In [15]:
kb_docs = [
    ("kb_python", "Python is a versatile programming language used in web development, data science, AI, and scripting. It has an extensive standard library and a large ecosystem of packages."),
    ("kb_rust",   "Rust is a systems programming language that guarantees memory safety without garbage collection. It's used for performance-critical applications like databases and operating systems."),
    ("kb_sochdb", "SochDB is a high-performance embedded database written in Rust. Features include vector search (HNSW), BM25 keyword search, ACID transactions, temporal graphs, and priority queues."),
    ("kb_rag",    "Retrieval-Augmented Generation (RAG) enhances LLM responses by retrieving relevant context from a knowledge base. It reduces hallucination and provides grounded answers."),
    ("kb_agents", "AI agents use LLMs with tool access to perform multi-step tasks autonomously. They can search databases, call APIs, manage memory, and reason about intermediate results."),
]

texts = [t for _, t in kb_docs]
vecs = embed_batch(texts)

for (doc_id, text), vec in zip(kb_docs, vecs):
    kb_collection.insert(id=doc_id, vector=vec, metadata={"text": text}, content=text)

print(f"Seeded {len(kb_docs)} knowledge base documents")

Seeded 5 knowledge base documents


## 3. LLM-Based Fact Extraction

We use Gemini to extract structured facts (entities + relations) from conversation text, then feed them into the consolidator.

In [16]:
def extract_facts_with_llm(text: str) -> dict:
    """Use Gemini to extract structured facts from text."""
    prompt = (
        "Extract structured facts from this text. Return ONLY valid JSON with this schema:\n"
        '{\n'
        '  "entities": [{"name": "...", "entity_type": "person|tool|concept|language|project", "properties": {}}],\n'
        '  "relations": [{"from_entity": "...", "relation_type": "...", "to_entity": "..."}],\n'
        '  "assertions": [{"subject": "...", "predicate": "...", "object": "...", "confidence": 0.9}]\n'
        '}\n\n'
        'Text: "' + text + '"\n\n'
        "Return ONLY the JSON, no markdown fences."
    )

    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
    )
    content = resp.choices[0].message.content.strip()
    # Remove markdown fencing if present
    if content.startswith("```"):
        content = content.split("\n", 1)[1]
        if content.endswith("```"):
            content = content[:-3]
    return json.loads(content)


def learn_from_text(text: str) -> dict:
    """
    Full memory pipeline:
    1. Extract facts with LLM
    2. Commit via extraction pipeline (writes to graph + vector store)
    3. Add to consolidator for deduplication
    """
    # 1. Extract
    raw = extract_facts_with_llm(text)
    
    # 2. Commit via extraction pipeline
    result = extraction_pipeline.extract(
        text=text,
        extractor=lambda t: raw,  # Use our pre-extracted results
        validate=False,
    )
    extraction_pipeline.commit(result)
    
    # 3. Add to consolidator
    for rel in result.relations:
        assertion = RawAssertion(
            id=rel.id,
            fact={"subject": rel.from_entity, "predicate": rel.relation_type, "object": rel.to_entity},
            embedding=embed(f"{rel.from_entity} {rel.relation_type} {rel.to_entity}"),
            confidence=rel.confidence,
            source=text[:100],
        )
        consolidator.add(assertion)
    
    for a in result.assertions:
        assertion = RawAssertion(
            id=a.id,
            fact={"subject": a.subject, "predicate": a.predicate, "object": str(a.object)},
            embedding=embed(f"{a.subject} {a.predicate} {a.object}"),
            confidence=a.confidence,
            source=text[:100],
        )
        consolidator.add(assertion)
    
    return {
        "entities": len(result.entities),
        "relations": len(result.relations),
        "assertions": len(result.assertions),
    }

print("Extraction + consolidation pipeline ready")

Extraction + consolidation pipeline ready


## 4. Conversational RAG + Memory Assistant

The assistant retrieves from BOTH the knowledge base AND memory, and learns from each conversation.

In [17]:
CACHE_NAME = "conv_rag_cache"
conversation_history = []

def chat(user_message: str) -> str:
    """
    Conversational RAG with memory:
    1. Check semantic cache
    2. Retrieve from knowledge base
    3. Retrieve from memory
    4. Generate response
    5. Learn new facts from the exchange
    6. Cache the response
    """
    print(f"\n{'='*60}")
    print(f"User: {user_message}")
    print(f"{'='*60}")
    
    # 1. Check semantic cache
    q_vec = embed(user_message)
    cached = db.cache_get(CACHE_NAME, q_vec, threshold=0.93)
    if cached:
        print("[Cache HIT]")
        return cached
    print("[Cache MISS]")
    
    # 2. Retrieve from knowledge base
    kb_response = kb_retriever.retrieve(
        query_text=user_message,
        query_vector=q_vec,
        allowed=AllowedSet.allow_all(),
        k=3,
    )
    kb_context = []
    for r in kb_response.results:
        meta = r.metadata or {}
        kb_context.append(meta.get("_content", meta.get("text", "")))
    print(f"[KB Retrieved: {len(kb_response.results)} docs]")
    
    # 3. Retrieve from memory
    mem_response = mem_retriever.retrieve(
        query_text=user_message,
        query_vector=q_vec,
        allowed=AllowedSet.allow_all(),
        k=3,
    )
    mem_context = []
    for r in mem_response.results:
        meta = r.metadata or {}
        mem_context.append(
            f"{meta.get('subject', '?')} {meta.get('predicate', '?')} {meta.get('object', '?')}"
        )
    print(f"[Memory Retrieved: {len(mem_response.results)} facts]")
    
    # 4. Also get consolidated canonical facts
    canonical_facts = list(consolidator.get_canonical_facts(current_only=True))
    canonical_str = ""
    if canonical_facts:
        facts_list = []
        for cf in canonical_facts[:5]:
            mf = cf.merged_fact
            facts_list.append(f"- {mf.get('subject','?')} {mf.get('predicate','?')} {mf.get('object','?')} (confidence: {cf.confidence:.2f})")
        canonical_str = "\n".join(facts_list)
    
    # 5. Build prompt
    system = (
        "You are a helpful assistant with access to a knowledge base and memory of past conversations. "
        "Use the provided context to answer accurately. If you remember facts from memory, mention that."
    )
    
    context_parts = []
    if kb_context:
        context_parts.append("KNOWLEDGE BASE:\n" + "\n".join(f"- {c}" for c in kb_context))
    if mem_context:
        context_parts.append("MEMORY (learned from past conversations):\n" + "\n".join(f"- {c}" for c in mem_context))
    if canonical_str:
        context_parts.append("CONSOLIDATED FACTS:\n" + canonical_str)
    
    context = "\n\n".join(context_parts) if context_parts else "No relevant context found."
    
    # Include recent conversation history
    history_str = ""
    if conversation_history:
        recent = conversation_history[-6:]  # Last 3 turns
        history_str = "\n\nRECENT CONVERSATION:\n" + "\n".join(
            f"{msg['role'].upper()}: {msg['content'][:200]}" for msg in recent
        )
    
    user_prompt = f"{context}{history_str}\n\nUser Question: {user_message}"
    
    # 6. Generate
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.3,
    )
    answer = resp.choices[0].message.content
    
    # 7. Learn from this exchange
    exchange = f"User said: {user_message}. Assistant answered: {answer[:300]}"
    try:
        learned = learn_from_text(exchange)
        print(f"[Learned: {learned}]")
    except Exception as e:
        print(f"[Learning skipped: {e}]")
    
    # 8. Cache the response
    db.cache_put(CACHE_NAME, user_message, answer, q_vec)
    print("[Cached response]")
    
    # 9. Update conversation history
    conversation_history.append({"role": "user", "content": user_message})
    conversation_history.append({"role": "assistant", "content": answer})
    
    # 10. Persist history to KV store
    history_key = b"conversation_history"
    db.put(history_key, json.dumps(conversation_history).encode())
    
    return answer

print("Conversational RAG + Memory assistant ready")

Conversational RAG + Memory assistant ready


## 5. Conversation Turn 1: Ask About a Topic

In [18]:
answer = chat("What is SochDB and what makes it special?")
print(f"\nAssistant: {answer}")


User: What is SochDB and what makes it special?
[Cache MISS]
[KB Retrieved: 3 docs]
[Memory Retrieved: 0 facts]
[Learning skipped: add() got an unexpected keyword argument 'id']
[Cached response]

Assistant: SochDB is a high-performance embedded database written in Rust.

What makes it special are its advanced features, including:

*   **Vector search (HNSW):** This allows for efficient similarity searches, which is crucial for applications like semantic search and recommendation systems.
*   **BM25 keyword search:** This provides traditional, robust keyword searching capabilities.
*   **ACID transactions:** This ensures data integrity and reliability, meaning operations are Atomic, Consistent, Isolated, and Durable.
*   **Temporal graphs:** This indicates support for managing and querying data that changes over time, useful for historical analysis or tracking evolving relationships.
*   **Priority queues:** This suggests efficient management of tasks or data based on their priority.


## 6. Conversation Turn 2: Tell the Agent Something New

In [19]:
answer = chat(
    "I'm building an AI application called SmartSearch that uses SochDB for vector storage. "
    "The project is written in Python and uses RAG for question answering."
)
print(f"\nAssistant: {answer}")


User: I'm building an AI application called SmartSearch that uses SochDB for vector storage. The project is written in Python and uses RAG for question answering.
[Cache MISS]
[KB Retrieved: 3 docs]
[Memory Retrieved: 0 facts]
[Learning skipped: add() got an unexpected keyword argument 'id']
[Cached response]

Assistant: That sounds like a very interesting project! Combining SochDB's vector search capabilities with Python and RAG for an AI application like SmartSearch is a smart approach.

Given your setup:

*   **SochDB:** Will handle the efficient storage and retrieval of your vector embeddings.
*   **Python:** Will be the primary language for developing your application logic, interacting with SochDB, and orchestrating the RAG process.
*   **RAG:** Will ensure that your AI's answers are grounded in the data stored in SochDB, reducing the likelihood of hallucinations and providing more accurate, contextually relevant responses.

This combination should allow you to build a powerful 

## 7. Conversation Turn 3: Test Memory Recall

In [20]:
answer = chat("What project am I building? What technology stack does it use?")
print(f"\nAssistant: {answer}")


User: What project am I building? What technology stack does it use?
[Cache MISS]
[KB Retrieved: 3 docs]
[Memory Retrieved: 0 facts]
[Learning skipped: add() got an unexpected keyword argument 'id']
[Cached response]

Assistant: You are building an AI application called **SmartSearch**.

The technology stack for SmartSearch includes:

*   **Python** for the application development.
*   **SochDB** for vector storage, leveraging its high-performance embedded database capabilities written in Rust.
*   **RAG (Retrieval-Augmented Generation)** for question answering.


## 8. Conversation Turn 4: Cache Hit Test

In [21]:
# Nearly identical question — should hit the cache
answer = chat("Tell me about SochDB and what makes it unique")
print(f"\nAssistant: {answer}")


User: Tell me about SochDB and what makes it unique
[Cache HIT]

Assistant: SochDB is a high-performance embedded database written in Rust.

What makes it special are its advanced features, including:

*   **Vector search (HNSW):** This allows for efficient similarity searches, which is crucial for applications like semantic search and recommendation systems.
*   **BM25 keyword search:** This provides traditional, robust keyword searching capabilities.
*   **ACID transactions:** This ensures data integrity and reliability, meaning operations are Atomic, Consistent, Isolated, and Durable.
*   **Temporal graphs:** This indicates support for managing and querying data that changes over time, useful for historical analysis or tracking evolving relationships.
*   **Priority queues:** This suggests efficient management of tasks or data based on their priority.

These features, particularly the combination of vector search and traditional keyword search, along with ACID compliance and tempor

## 9. Inspect Consolidated Memory

View the canonical facts that the consolidator has built from all conversations.

In [22]:
# Run consolidation to merge/deduplicate raw assertions
updated = consolidator.consolidate()
print(f"Consolidation updated {updated} canonical facts\n")

# Get all canonical facts from the consolidator
print("=== Consolidated Canonical Facts ===")
facts = list(consolidator.get_canonical_facts(current_only=True))
if facts:
    for i, fact in enumerate(facts, 1):
        mf = fact.merged_fact
        print(f"  {i}. {mf.get('subject','?')} → {mf.get('predicate','?')} → {mf.get('object','?')}")
        print(f"     confidence={fact.confidence:.2f}, support_count={len(fact.support_set)}")
else:
    print("  (No canonical facts yet — consolidation may need more data)")

# Show raw assertions via backend scan
raw_assertions = list(consolidator._backend.scan_raw_assertions(consolidator._namespace))
print(f"\nTotal raw assertions: {len(raw_assertions)}")
for a in raw_assertions[:10]:
    f = a.fact
    print(f"  - {f.get('subject','?')} {f.get('predicate','?')} {f.get('object','?')}")

Consolidation updated 0 canonical facts

=== Consolidated Canonical Facts ===
  (No canonical facts yet — consolidation may need more data)

Total raw assertions: 0


## 10. Inspect Persisted Conversation History

In [23]:
# Retrieve conversation history from KV store
history_raw = db.get(b"conversation_history")
if history_raw:
    history = json.loads(history_raw.decode())
    print(f"Stored {len(history)} messages in conversation history:")
    for msg in history:
        role = msg['role'].upper()
        content = msg['content'][:120]
        print(f"  [{role}] {content}..." if len(msg['content']) > 120 else f"  [{role}] {content}")

Stored 6 messages in conversation history:
  [USER] What is SochDB and what makes it special?
  [ASSISTANT] SochDB is a high-performance embedded database written in Rust.

What makes it special are its advanced features, includ...
  [USER] I'm building an AI application called SmartSearch that uses SochDB for vector storage. The project is written in Python ...
  [ASSISTANT] That sounds like a very interesting project! Combining SochDB's vector search capabilities with Python and RAG for an AI...
  [USER] What project am I building? What technology stack does it use?
  [ASSISTANT] You are building an AI application called **SmartSearch**.

The technology stack for SmartSearch includes:

*   **Python...


## 11. Cleanup

In [24]:
db.close()
shutil.rmtree(DB_PATH, ignore_errors=True)
print("Done! Database cleaned up.")

Done! Database cleaned up.
